# PA2 Task 1: Repository and Kaggle setup

Clones the repo, installs dependencies without reinstalling PyTorch, confirms two T4 GPUs, prints package versions, and runs the public adapter tests.

The repo is private: add a Kaggle secret named `GITHUB_TOKEN` (Add-ons > Secrets) with read access to it.

In [ ]:
REPO_URL = "https://github.com/usmanbinnaeem/small-model-alignment-starter.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

In [ ]:
!nvidia-smi

In [ ]:
import os, subprocess

clone_url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
except Exception:
    pass  # public repo, or no secret attached

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, clone_url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!git log -1 --oneline

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest peft trl
!python -m pip install -q -e . --no-deps

In [ ]:
import torch, transformers, datasets, peft, trl, sys

print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("peft        ", peft.__version__)
print("trl         ", trl.__version__)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}  {p.name}  {p.total_memory / 2**30:.1f} GiB")
assert torch.cuda.device_count() == 2, "Expected GPU T4 x2

In [ ]:
!PYTHONPATH=. python -m pytest -q tests/test_public_adapter_smoke.py